# D3 · Notebook para grabar el video
**Álvaro Contreras y Pablo Cortés · Ingeniería UdeC**

Ejecuta **A1–A5 antes de grabar**. Después graba **B0–B4**, en orden. Al terminar, ejecuta **C1** y descarga la evidencia.

- GPU T4. No se ejecuta el lote de 75 preguntas ni se modifica el sistema v6.
- B1 y B2 llaman realmente a Qwen3-4B. B3 muestra un caso histórico y B4 cifras históricas provisionales.
- Preparación y descarga de pesos quedan fuera del video. El guion está en `D3 alvaro/Guion_video_D3.md`.
- Si la inferencia supera el tiempo disponible, recorta únicamente esperas en la edición y señala **«espera recortada»**; conserva los mensajes, el resultado y los tiempos reales. Máximo del video: 3 minutos.
- Cada intento se guarda por separado. Descarga la evidencia antes de cerrar Colab.


## A1 · Preparar dependencias — fuera de la grabación
Usa un entorno nuevo con GPU: **Entorno de ejecución → Cambiar tipo de entorno → T4**.
No vuelvas a ejecutar esta instalación después de cargar los modelos. Si Colab solicita reiniciar, hazlo y continúa desde A2.

Se conserva la rama 4 de Transformers compatible con este código. Las versiones exactas resueltas se guardan en la evidencia; no se afirma que sean idénticas a las de la evaluación histórica.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.51,<5', 'accelerate>=1,<2', 'bitsandbytes>=0.45,<1',
    'sentence-transformers>=4,<6'])
print('Dependencias instaladas. Si Colab pide reiniciar, reinicia y continúa desde A2.')


## A2 · Cargar la versión del repositorio — fuera de la grabación
La opción predeterminada descarga el paquete de `D3 alvaro` y verifica su SHA-256 contra el manifiesto del mismo commit. No necesita credenciales.

Si falla la descarga, cambia `DESCARGAR_DEL_REPO = False` y sube `D3_alvaro_ejecucion.zip`. No subas el ZIP original de Álvaro: esta demo necesita el módulo de registro nuevo.


In [ ]:
from pathlib import Path
import sys, json, hashlib, tempfile, zipfile, urllib.request, stat
import torch
assert torch.cuda.is_available(), 'Activa la GPU T4 antes de continuar.'
print('GPU:', torch.cuda.get_device_name(0))
DESCARGAR_DEL_REPO = True
FUENTE_REF = 'main'  # también puede ser un commit concreto
REPO = 'Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1'

def leer_url(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'D3-video-Colab'})
    with urllib.request.urlopen(req, timeout=180) as respuesta:
        return respuesta.read()

commit_fuentes = None
if DESCARGAR_DEL_REPO:
    commit_fuentes = json.loads(leer_url(f'https://api.github.com/repos/{REPO}/commits/{FUENTE_REF}'))['sha']
    raw = f'https://raw.githubusercontent.com/{REPO}/{commit_fuentes}/D3%20alvaro/'
    datos_zip = leer_url(raw + 'D3_alvaro_ejecucion.zip')
    manifiesto = json.loads(leer_url(raw + 'auditoria/manifiesto.json'))
    assert hashlib.sha256(datos_zip).hexdigest() == manifiesto['D3_alvaro_ejecucion.zip'], 'Hash del paquete incorrecto.'
else:
    from google.colab import files
    subidos = files.upload()
    assert len(subidos) == 1, 'Sube únicamente D3_alvaro_ejecucion.zip.'
    nombre, datos_zip = next(iter(subidos.items()))
    assert nombre.endswith('.zip'), 'El archivo debe ser ZIP.'

paquete_sha256 = hashlib.sha256(datos_zip).hexdigest()
raiz = Path(tempfile.mkdtemp(prefix='d3_video_', dir='/content'))
paquete = raiz/'paquete.zip'
paquete.write_bytes(datos_zip)
with zipfile.ZipFile(paquete) as z:
    for item in z.infolist():
        destino = (raiz/item.filename).resolve()
        assert destino.is_relative_to(raiz.resolve()), 'Ruta no válida en ZIP.'
        assert not stat.S_ISLNK(item.external_attr >> 16), 'No se admiten enlaces simbólicos.'
    z.extractall(raiz)
BASE = raiz/'Deliverable3'
assert (BASE/'video/soporte_video.py').is_file(), 'Usa el paquete actualizado de D3 alvaro.'
AGENTES = BASE/'agentes'
for ruta in (AGENTES, BASE/'video'):
    sys.path.insert(0, str(ruta))
for modulo in ('orquestador','tools','soporte_video'):
    sys.modules.pop(modulo, None)
print('Fuentes:', BASE)
print('Commit:', commit_fuentes or 'archivo subido; se registra su hash')
print('SHA-256 del ZIP:', paquete_sha256)


## A3 · Cargar Qwen3-4B — fuera de la grabación
Puede tardar varios minutos. Todos los roles comparten estos pesos. Se mantienen los parámetros y presupuestos de salida de v6; la demo no reduce el razonamiento para acelerar el video.


In [ ]:
import hashlib
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
MODELO = 'Qwen/Qwen3-4B'
REVISION_MODELO = model_info(MODELO).sha
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODELO, revision=REVISION_MODELO)
model = AutoModelForCausalLM.from_pretrained(MODELO, revision=REVISION_MODELO,
    quantization_config=bnb, device_map='auto')
model.eval()
MAX_TOKENS = {'enrutador':256, 'recuperador':160, 'redactor':1200, 'verificador':700}
SEMILLA = 2026
PARAMETROS = dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20, min_p=0.0)

def generar_real(agente, prompt):
    texto = tok.apply_chat_template([{'role':'user','content':prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    semilla = SEMILLA + int(hashlib.sha256((agente+prompt).encode()).hexdigest()[:8],16) % 1_000_000
    torch.manual_seed(semilla)
    entradas = tok(texto, return_tensors='pt').to(model.device)
    with torch.no_grad():
        salida = model.generate(**entradas, max_new_tokens=MAX_TOKENS[agente],
            pad_token_id=tok.eos_token_id, **PARAMETROS)
    return tok.decode(salida[0][entradas.input_ids.shape[1]:], skip_special_tokens=True).strip()

print('Modelo listo:', MODELO, '| revisión:', REVISION_MODELO, '| cómputo:', DTYPE)


## A4 · Preparar la recuperación E5 — fuera de la grabación
E5 es el modelo de embeddings de la herramienta de búsqueda. No es un agente generativo adicional.


In [ ]:
from tools import Corpus
from sentence_transformers import SentenceTransformer
import numpy as np
MODELO_E5 = 'intfloat/multilingual-e5-small'
REVISION_E5 = model_info(MODELO_E5).sha
corpus = Corpus(AGENTES)
e5 = SentenceTransformer(MODELO_E5, revision=REVISION_E5, device='cuda')
fragmentos = corpus.frag
vectores = e5.encode([f['texto_embedding'] for f in fragmentos],
    normalize_embeddings=True, batch_size=64, show_progress_bar=True)

def buscar_e5(consulta, k=5):
    q = e5.encode(['query: '+str(consulta)], normalize_embeddings=True)[0]
    orden = np.argsort(-(vectores @ q))
    vistos, encontrados = set(), []
    for i in orden:
        f = fragmentos[int(i)]
        uid = f.get('unidad_id') or f.get('parent_id')
        if uid in vistos:
            continue
        vistos.add(uid)
        u = corpus.por_id.get(uid, {})
        encontrados.append({'id':uid, 'texto':u.get('texto', f['texto'])})
        if len(encontrados) >= k:
            break
    return encontrados

corpus.buscar = buscar_e5
print('E5 listo:', len(corpus.por_id), 'unidades y', len(fragmentos), 'fragmentos.')


## A5 · Dejar lista la grabación — fuera de la grabación
Esta celda comprueba archivos y prepara el registro, pero no ejecuta las preguntas. Si ensayas B1/B2, cada intento tendrá un identificador diferente. No vuelvas a ejecutar A5 entre el ensayo y la grabación: conserva la lista de intentos para exportarla.


In [ ]:
import csv, platform, importlib.metadata
from soporte_video import (SesionDemo, mostrar_portada, mostrar_fallo,
                           mostrar_resultados, exportar_evidencia)
versiones = {p:importlib.metadata.version(p) for p in
    ('torch','transformers','accelerate','bitsandbytes','sentence-transformers',
     'huggingface-hub','tokenizers','numpy','safetensors')}
assert importlib.metadata.version('transformers') == __import__('transformers').__version__, 'Reinicia el entorno: versión importada distinta de la instalada.'
ENTORNO = {'python':platform.python_version(), 'gpu':torch.cuda.get_device_name(0),
    'cuda':torch.version.cuda, 'versiones':versiones, 'modelo':MODELO,
    'revision_modelo':REVISION_MODELO, 'embeddings':MODELO_E5, 'revision_e5':REVISION_E5,
    'cuantizacion':'NF4 con doble cuantización', 'dtype':str(DTYPE),
    'parametros':PARAMETROS, 'max_tokens':MAX_TOKENS, 'enable_thinking':False,
    'semilla_base':SEMILLA, 'regla_semilla':'base + int(sha256(agente+prompt)[:8],16) % 1000000',
    'commit_fuentes':commit_fuentes, 'paquete_sha256':paquete_sha256,
    'fuentes_sha256':{str(p.relative_to(BASE)):hashlib.sha256(p.read_bytes()).hexdigest()
       for carpeta in ('agentes','prompts','video') for p in sorted((BASE/carpeta).rglob('*'))
       if p.is_file() and '__pycache__' not in p.parts and p.suffix in ('.py','.txt','.json','.csv')}}
with (AGENTES/'test_set_50.csv').open(encoding='utf-8-sig', newline='') as f:
    preguntas = {str(r['id']):r['pregunta'] for r in csv.DictReader(f)}
assert '1' in preguntas and '41' in preguntas
EVIDENCIA = Path('/content/D3_video_evidencia')
EVIDENCIA.mkdir(exist_ok=True)
sesiones_video = []
ultima_por_caso = {}
print('LISTO PARA GRABAR. Empieza en B0 y sigue hasta B4.')
print('Caso 1:', preguntas['1'])
print('Caso 41:', preguntas['41'])
print('Al terminar, C1 descarga prompts, salidas completas, herramientas y entorno.')


# B · Grabación — empieza aquí
Cierra el panel lateral y usa un zoom que permita leer los mensajes. Ejecuta cada celda individualmente; **no uses «Ejecutar todo»** durante la grabación.


In [ ]:
# B0 · Portada y arquitectura — unos 15 segundos
mostrar_portada()


## B1 · Consulta normativa real
Ejecuta la celda al empezar a explicar el recorrido. Verás la decisión de cada agente cuando termine su llamada. No anticipes una respuesta que todavía no haya aparecido.


In [ ]:
sesion_1 = SesionDemo(corpus, generar_real, EVIDENCIA, ENTORNO,
    caso='P01', pregunta=preguntas['1'])
sesiones_video.append(sesion_1)
ultima_por_caso['P01'] = sesion_1.id
resultado_1 = sesion_1.ejecutar()


## B2 · Artículo inexistente: segunda consulta real
Muestra la decisión del Enrutador y la comprobación de existencia. Si la salida difiere del histórico, explica lo que realmente aparece.


In [ ]:
sesion_2 = SesionDemo(corpus, generar_real, EVIDENCIA, ENTORNO,
    caso='P41', pregunta=preguntas['41'])
sesiones_video.append(sesion_2)
ultima_por_caso['P41'] = sesion_2.id
resultado_2 = sesion_2.ejecutar()


## B3 · Fallo histórico H03 — no ejecuta el modelo
Explica por qué la revisión puede detectar un problema sin resolverlo. Esta pantalla distingue el primer borrador de la respuesta final.


In [ ]:
fallo_historico = mostrar_fallo(BASE)


## B4 · Resultados y cierre — no ejecuta el modelo
La tabla corresponde a la evaluación histórica propuesta, no a los dos ejemplos del video.


In [ ]:
mostrar_resultados()


# C · Después de grabar
## C1 · Descargar evidencia y conservarla en el repositorio
La selección predeterminada usa el último intento de cada caso. **Comprueba que estos identificadores coincidan con la grabación.** El ZIP conserva todos los intentos de esta sesión y distingue los seleccionados.

Guarda el ZIP descargado. Después se debe subir a `D3 alvaro/evidencia_video/`, publicar el video con acceso abierto y añadir el enlace al informe. El notebook no publica archivos en tu nombre.


In [ ]:
RUN_IDS_EN_VIDEO = [ultima_por_caso[c] for c in ('P01','P41') if c in ultima_por_caso]
for s in sesiones_video:
    print('EN VIDEO' if s.id in RUN_IDS_EN_VIDEO else 'otro intento',
          s.id, s.datos['caso'], s.datos['estado'], s.datos['pregunta'])
# Si usaste otro intento, reemplaza RUN_IDS_EN_VIDEO por sus identificadores exactos.
paquete_evidencia = exportar_evidencia(BASE, EVIDENCIA, sesiones_video, RUN_IDS_EN_VIDEO, ENTORNO)
print('Evidencia guardada:', paquete_evidencia)
from google.colab import files
files.download(str(paquete_evidencia))


## C2 · Inspección opcional de un mensaje completo
Fuera de la grabación, puedes consultar cualquier prompt y salida cruda. Los mismos datos están en el ZIP; la visualización resumida no elimina información del registro nuevo.


In [ ]:
SESION_A_REVISAR = sesiones_video[-1]  # cambia por sesion_1 si lo necesitas
NUMERO_LLAMADA = 1
llamada = SESION_A_REVISAR.datos['llamadas'][NUMERO_LLAMADA-1]
print('AGENTE:', llamada['agente'])
print('PROMPT COMPLETO:\n', llamada['mensajes'][0]['content'])
print('\nSALIDA CRUDA:\n', llamada.get('salida_cruda', '(llamada incompleta)'))


## Referencias técnicas y alcance
- [Modelo Qwen3-4B y configuración sin thinking](https://huggingface.co/Qwen/Qwen3-4B).
- [Cuantización con bitsandbytes](https://huggingface.co/docs/transformers/quantization/bitsandbytes).
- [Sistema, fuentes y evaluación](https://github.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1/tree/main/D3%20alvaro).

Esta instrumentación añade visualización y registro a v6. No cambia prompts, decisiones, corpus ni presupuestos de generación. Las pruebas locales usan un modelo simulado para verificar el registro; la ejecución GPU debe comprobarse en Colab. La duración y las respuestas no están garantizadas entre entornos.
